In [1]:
import sys
import time
sys.path.append('/home/pi/project_demo/lib')
from McLumk_Wheel_Sports import *

speed = 20

# metrics
bias_values = []
lost_line_count = 0
prev_lost_line = False
recovery_times = []
recovery_start_time = None
is_recovering = False

try:
    while True:
        # Read line sensor data
        track_data = bot.read_data_array(0x0a, 1)
        track = int(track_data[0])

        # Decode sensor states
        x1 = (track >> 3) & 0x01
        x2 = (track >> 2) & 0x01
        x3 = (track >> 1) & 0x01
        x4 = track & 0x01

        lineL1 = x2
        lineL2 = x1
        lineR1 = x3
        lineR2 = x4

        # -------------------------
        # 1. IR-based bias estimate
        # -------------------------
        active_positions = []

        if lineL1 == 0:
            active_positions.append(-3)
        if lineL2 == 0:
            active_positions.append(-1)
        if lineR1 == 0:
            active_positions.append(1)
        if lineR2 == 0:
            active_positions.append(3)

        if len(active_positions) > 0:
            ir_bias = sum(active_positions) / len(active_positions)
            bias_values.append(abs(ir_bias))
        else:
            ir_bias = None

        # -------------------------
        # 2. Lost line detection
        # -------------------------
        if lineL1 == 1 and lineL2 == 1 and lineR1 == 1 and lineR2 == 1:
            current_lost_line = True
        else:
            current_lost_line = False

        if current_lost_line and not prev_lost_line:
            lost_line_count += 1
            recovery_start_time = time.time()
            is_recovering = True

        if (not current_lost_line) and prev_lost_line and is_recovering:
            recovery_time = time.time() - recovery_start_time
            recovery_times.append(recovery_time)
            recovery_start_time = None
            is_recovering = False

        prev_lost_line = current_lost_line

        # -------------------------
        # Original control logic
        # -------------------------
        if lineL1 == 0 and lineL2 == 0 and lineR1 == 0 and lineR2 == 0:
            #print("1")
            #print(lineL1, lineL2, lineR1, lineR2)
            move_forward(int(speed))

        elif ((lineL2 == 0 or lineL1 == 0) and lineR2 == 0):
            #print("2")
            #print(lineL1, lineL2, lineR1, lineR2)
            rotate_right(speed)
            time.sleep(0.05)

        elif lineL1 == 0 and (lineR2 == 0 or lineR1 == 0):
            #print("3")
            #print(lineL1, lineL2, lineR1, lineR2)
            rotate_left(int(speed * 1.5))
            time.sleep(0.15)

        elif lineL1 == 0:
            #print("4")
            #print(lineL1, lineL2, lineR1, lineR2)
            rotate_left(speed)
            time.sleep(0.02)

        elif lineR2 == 0:
            #print("5")
            #print(lineL1, lineL2, lineR1, lineR2)
            rotate_right(speed)
            time.sleep(0.01)

        elif lineL2 == 0 and lineR1 == 1:
            #print("6")
            #print(lineL1, lineL2, lineR1, lineR2)
            rotate_left(int(speed))

        elif lineL2 == 1 and lineR1 == 0:
            #print("7")
            #print(lineL1, lineL2, lineR1, lineR2)
            rotate_right(int(speed))

        elif lineL2 == 0 and lineR1 == 0:
            #print("8")
            #print(lineL1, lineL2, lineR1, lineR2)
            move_forward(speed)

        time.sleep(0.01)

except KeyboardInterrupt:
    stop()

    if len(bias_values) > 0:
        mean_abs_bias = sum(bias_values) / len(bias_values)
    else:
        mean_abs_bias = 0

    if len(recovery_times) > 0:
        avg_recovery_time = sum(recovery_times) / len(recovery_times)
    else:
        avg_recovery_time = 0

    print("===== RUN RESULTS =====")
    print(f"Mean |Bias|: {mean_abs_bias:.2f}")
    print(f"Lost Line Count: {lost_line_count}")
    print(f"Average Recovery Time: {avg_recovery_time:.2f} s")
    print("=======================")
    print("Ending")


===== RUN RESULTS =====
Mean |Bias|: 0.28
Lost Line Count: 57
Average Recovery Time: 0.14 s
Ending


In [2]:
# 清理资源 Release resources
stop()
del bot